<a href="https://www.kaggle.com/code/avikdas567/predicting-heart-disease?scriptVersionId=300737780" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier
import lightgbm as lgb

# Load Data

train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

TARGET = "Heart Disease"
ID_COL = "id"

X = train.drop(columns=[TARGET])
y = train[TARGET].map({"Absence": 0, "Presence": 1}).astype(int)

X_test = test.copy()

# Encode Categorical Features

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    full = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(full)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# Settings

N_SPLITS = 7
SEED = 42

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

print("Training started...\n")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    
    print(f"Fold {fold}")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    cat_model = CatBoostClassifier(
        iterations=3500,
        learning_rate=0.025,
        depth=7,
        l2_leaf_reg=6,
        border_count=128,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        task_type="GPU",
        devices="0",
        early_stopping_rounds=200,
        verbose=False
    )
    
    cat_model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    
    val_cat = cat_model.predict_proba(X_val)[:, 1]
    test_cat = cat_model.predict_proba(X_test)[:, 1]
    
    lgb_params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.03,
        "num_leaves": 80,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "lambda_l1": 0.5,
        "lambda_l2": 2.0,
        "device": "gpu",
        "gpu_platform_id": 0,
        "gpu_device_id": 0,
        "seed": SEED,
        "verbosity": -1
    }
    
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_valid = lgb.Dataset(X_val, y_val)
    
    lgb_model = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=5000,
        valid_sets=[lgb_valid],
        callbacks=[
            lgb.early_stopping(250),
            lgb.log_evaluation(0)
        ]
    )
    
    val_lgb = lgb_model.predict(X_val)
    test_lgb = lgb_model.predict(X_test)
    
    val_pred = 0.65 * val_cat + 0.35 * val_lgb
    test_pred = 0.65 * test_cat + 0.35 * test_lgb
    
    oof_preds[val_idx] = val_pred
    test_preds += test_pred / N_SPLITS
    
    fold_auc = roc_auc_score(y_val, val_pred)
    print(f"  AUC: {fold_auc:.6f}")

# Final CV

cv_auc = roc_auc_score(y, oof_preds)
print("\n============================")
print(f"FINAL CV ROC-AUC: {cv_auc:.6f}")
print("============================\n")

# Submission

submission = pd.DataFrame({
    "id": test[ID_COL],
    "Heart Disease": test_preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved successfully.")
display(submission.head())

Training started...

Fold 1


Default metric period is 5 because AUC is/are not implemented for GPU
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[599]	valid_0's auc: 0.955617
  AUC: 0.955746
Fold 2


Default metric period is 5 because AUC is/are not implemented for GPU


Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[528]	valid_0's auc: 0.955002
  AUC: 0.955127
Fold 3


...

Default metric period is 5 because AUC is/are not implemented for GPU


Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[663]	valid_0's auc: 0.956409
  AUC: 0.956487
Fold 7


Default metric period is 5 because AUC is/are not implemented for GPU


Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[759]	valid_0's auc: 0.955393
  AUC: 0.955444

FINAL CV ROC-AUC: 0.955316

submission.csv saved successfully.


,id,Heart Disease
0,630000,0.940729
1,630001,0.007307
2,630002,0.986580
3,630003,0.003617
4,630004,0.177605
